## Bitboard

In [362]:
class BitBoard:
    def __init__(self, width, height, dataWidth, state = 0):
        self.width = width
        self.height = height
        self.dataWidth = dataWidth
        self.state = state
    
    def inBounds(self, x, y):
        return x >= 0 and x < self.width and y >= 0 and y < self.height
    
    def getXY(self, i):
        x = i % self.width
        y = self.height - 1 - (i // self.width)

        if(self.inBounds(x, y)):
            return x, y
        else:
            return -1, -1

    def getI(self, x, y):
        if(self.inBounds(x, y)):
            return (self.height - 1 - y) * self.width + x
        else:
            return -1
        
    def getAt(self, i):
        return (self.state >> (i * self.dataWidth)) & ((1 << self.dataWidth) - 1)
    
    def getAt2D(self, x, y):
        i = self.getI(x, y)
        if(i == -1):
            return None

        return self.getAt(i)

    def setAt(self, i, value):
        mask = ((1 << self.dataWidth) - 1) << i * self.dataWidth
        self.state &= ~mask
        self.state |= value << i * self.dataWidth

    def setAt2D(self, x, y, value):
        i = self.getI(x, y)
        if(i == -1):
            return

        return self.setAt(i, value)

class ChessBoardMask(BitBoard):
    def __init__(self, state = 0):
        BitBoard.__init__(self, 8, 8, 1, state)
    
    def flipAt2D(self, x, y):
        super().setAt2D(x, y, 0 if self.getAt2D(x, y) else 1)

    def flipAt(self, i):
        super().setAt(i, 0 if self.getAt(i) else 1)
    
    def getAt(self, i):
        return True if super().getAt(i) == 1 else False
    
    def setAt(self, i, value):
        super().setAt(i, 1 if value else 0)
    
    def setAt2D(self, x, y, value):
        super().setAt2D(x, y, 1 if value else 0)
    
    def getAt2D(self, x, y):
        return True if super().getAt2D(x, y) == 1 else False

    def getOr(self, mask):
        return ChessBoardMask(self.state | mask.state)

    def getAnd(self, mask):
        return ChessBoardMask(self.state & mask.state)

    def getIncludes(self, mask):
        return (self.state & mask.state) == mask.state
    
    def validPositions(self):
        for y in range(self.height):
            for x in range(self.width):
                if(self.getAt2D(x,y)):
                    yield (x, y)

    def __str__(self):
        box_width = 3

        string = ""
        string += " " + " ".center(box_width) + " "
        string += "┌"
        for x in range(self.width - 1):
            string += "─" * box_width + "┬"
        string += "─" * box_width + "┐\n"

        for y in range(self.height):

            string += " "
            string += str(self.width - y).center(box_width) + " "

            string += "│"
            for x in range(self.width):
                value = self.getAt2D(x, self.height - y - 1)
                symbol = "X" if value else " "
                string += symbol.center(box_width) + "│"

            string += "\n " + " ".center(box_width) + " "

            if(y < self.height - 1):
                string += "├" + (("─" * box_width + "┼") * (self.width - 1)) + ("─" * box_width + "┤")
            else:
                string += "└" + (("─" * box_width + "┴") * (self.width - 1)) + ("─" * box_width + "┘")

            string += "\n"
        
        string += "  " + " ".center(box_width) + " "
        for letter in ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h']:
            string += letter.center(box_width) + " "

        return string

class ChessBoardState(BitBoard):
    def __init__(self, state = 0):
        BitBoard.__init__(self, 8, 8, 4, state)
    
    def copyWithMove(self, x, y, newX, newY):
        copy = ChessBoardState(self.state)
        val = copy.getAt2D(x, y)
        copy.setAt2D(x, y, PieceDefs.EMPTY.getValue(True))
        copy.setAt2D(newX, newY, val)
        return copy

    def __str__(self):
        box_width = 3

        string = ""
        string += " " + " ".center(box_width) + " "
        string += "┌"
        for x in range(self.width - 1):
            string += "─" * box_width + "┬"
        string += "─" * box_width + "┐\n"

        for y in range(self.height):

            string += " "
            string += str(self.width - y).center(box_width) + " "

            string += "│"
            for x in range(self.width):
                pieceValue = self.getAt2D(x, self.height - y - 1)
                piece = PieceDefs.VALUE_MAP[Piece.extractPiece(pieceValue)]
                symbol = piece.getSymbol(Piece.isWhite(pieceValue))
                string += symbol.center(box_width) + "│"

            string += "\n " + " ".center(box_width) + " "

            if(y < self.height - 1):
                string += "├" + (("─" * box_width + "┼") * (self.width - 1)) + ("─" * box_width + "┤")
            else:
                string += "└" + (("─" * box_width + "┴") * (self.width - 1)) + ("─" * box_width + "┘")

            string += "\n"
        
        string += "  " + " ".center(box_width) + " "
        for letter in ['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h']:
            string += letter.center(box_width) + " "

        return string

## Pieces & Moves

In [363]:
class Piece:
    WHITE_VALUE = 0b0000
    BLACK_VALUE = 0b1000
    
    @staticmethod
    def isWhite(coloredPieceValue):
        return Piece.extractColor(coloredPieceValue) == 0
    
    @staticmethod
    def extractColor(coloredPieceValue):
        return (coloredPieceValue & 0b1000)

    @staticmethod
    def extractPiece(coloredPieceValue):
        return (coloredPieceValue & 0b0111)

    def __init__(self, name, symbols, value, moveDefinitions):
        self.name = name
        self.symbols = symbols
        self.value = value
        self.moveDefinitions = moveDefinitions

    def __str__(self):
        return self.name

    def getValidMoves(self, state, position, onlyCapturing = False): # only return moves that CAN capture (not WILL)
        isWhite = Piece.isWhite(state.getAt(position))
        x, y = state.getXY(position)

        if(x == -1 or y == -1):
            raise Exception("Attempted to solve moves for piece at invalid position index " + position + ".")

        mask = ChessBoardMask()

        for move in self.moveDefinitions:
            if(onlyCapturing and not move.canCapture):
                continue

            possibles = move.getPossibleFrom(x, y, not isWhite)
            
            for direction in possibles:
                positions = possibles[direction]
                for position in positions:
                    value = state.getAt2D(position[0], position[1])

                    if(PieceDefs.isEmpty(value) and not move.mustCapture):
                        mask.setAt2D(position[0], position[1], True)
                    elif(not PieceDefs.isEmpty(value)):
                        if(move.canCapture and Piece.extractColor(value) != Piece.WHITE_VALUE if isWhite else Piece.BLACK_VALUE):
                            mask.setAt2D(position[0], position[1], True)
                        if(not move.jump): # Because custom pieces aren't exist, 'jump' also messes with move generation
                            break # Cannot move over piece
        return mask
    def getSymbol(self, asWhite):
        return self.symbols[0 if asWhite else 1]
    def getValue(self, asWhite):
        return self.value | (Piece.WHITE_VALUE if asWhite else Piece.BLACK_VALUE)
    def isValue(self, value):
        return (value & 0b0111) == self.value

class MoveDef:
    def __init__(self, direction, strength = -1, mirrorX = False, mirrorY = False, jump = False, canCapture = False, mustCapture = False):
        self.boundsX = 8
        self.boundsY = 8
        
        self.direction = direction
        self.strength = strength
        self.mirrorX = mirrorX
        self.mirrorY = mirrorY
        self.jump = jump
        self.canCapture = canCapture
        self.mustCapture = mustCapture
    def getPossibleFrom(self, x, y, flipY = False):
        possible = {}

        if(not self.jump):
            directions = [(self.direction[0], self.direction[1] if not flipY else -1 * self.direction[1])]
            root_dir = directions[0]
            if(self.mirrorX):
                directions.append((root_dir[0] * -1, root_dir[1]))
            if(self.mirrorY):
                directions.append((root_dir[0], root_dir[1] * - 1))
            if(self.mirrorX and self.mirrorY):
                directions.append((root_dir[0] * -1, root_dir[1] * - 1))

            for direction in directions:
                possible[direction] = []
                # This little ternary (and only it) removes support of asymmetrical boards
                strengthI = self.boundsX if self.strength <= 0 else self.strength

                for i in range(1, strengthI + 1):
                    curX = x + (direction[0] * i)
                    curY = y + (direction[1] * i)
                    if(not self.inBounds(curX, curY)):
                        break
                    possible[direction].append((curX, curY))
        elif(self.jump):
            directions = [(self.direction[0], self.direction[1] if not flipY else -1 * self.direction[1])]
            root_dir = directions[0]
            if(self.mirrorX):
                directions.append((root_dir[0] * -1, root_dir[1]))
            if(self.mirrorY):
                directions.append((root_dir[0], root_dir[1] * - 1))
            if(self.mirrorX and self.mirrorY):
                directions.append((root_dir[0] * -1, root_dir[1] * - 1))

            for direction in directions:
                possible[direction] = []
                pos = (x + direction[0], y + direction[1])
                if(self.inBounds(pos[0], pos[1])):
                    possible[direction].append(pos)

        return possible
    def inBounds(self, x, y):
        return x >= 0 and x < self.boundsX and y >= 0 and y < self.boundsY

class PieceDefs:
    EMPTY_VALUE = 0b000
    PAWN_VALUE = 0b001
    KNIGHT_VALUE = 0b010
    BISHOP_VALUE = 0b011
    ROOK_VALUE = 0b100
    QUEEN_VALUE = 0b101
    KING_VALUE = 0b110

    EMPTY = Piece("", ("  ", "  "), EMPTY_VALUE, [])
    PAWN = Piece("Pawn", ("P", "p"), PAWN_VALUE, [
        MoveDef((0,1), 1, False, False, False, False, False),
        MoveDef((1,1), 1, True, False, False, True, True),
    ])
    KNIGHT = Piece("Knight", ("N", "n"), KNIGHT_VALUE, [
        MoveDef((2, 1), -1, True, True, True, True, False)
    ])
    BISHOP = Piece("Bishop", ("B", "b"), BISHOP_VALUE, [
        MoveDef((1,1), -1, True, True, False, True, False)
    ])
    ROOK = Piece("Rook", ("R", "r"), ROOK_VALUE, [
        MoveDef((1,0), -1, True, False, False, True, False),
        MoveDef((0,1), -1, False, True, False, True, False)
    ])
    QUEEN = Piece("Queen", ("Q", "q"), QUEEN_VALUE, [
        MoveDef((1,0), -1, True, False, False, True, False),
        MoveDef((0,1), -1, False, True, False, True, False),
        MoveDef((1,1), -1, True, True, False, True, False)
    ])
    KING = Piece("King", ("K", "k"), KING_VALUE, [
        MoveDef((1,0), 1, True, False, False, True, False),
        MoveDef((0,1), 1, False, True, False, True, False),
        MoveDef((1,1), 1, True, True, False, True, False)
    ])
    
    VALUE_MAP = {
        EMPTY_VALUE: EMPTY,
        PAWN_VALUE: PAWN,
        KNIGHT_VALUE: KNIGHT,
        BISHOP_VALUE: BISHOP,
        ROOK_VALUE: ROOK,
        QUEEN_VALUE : QUEEN,
        KING_VALUE : KING
    }

    @staticmethod
    def isEmpty(value):
        return value == PieceDefs.EMPTY.getValue(True) or value == PieceDefs.EMPTY.getValue(False)


## Evaluations

In [364]:
def getPieces(board, filter = None):
    pieces = [] # (x, y), piece
    for i in range(board.width * board.height):
        value = board.getAt(i)
        if(not PieceDefs.isEmpty(value)):
            if(filter == None or Piece.extractColor(value) == filter):
                pieces.append((board.getXY(i), value))
    return pieces

def getMoves(board, onlyCapturing = False, filter = None):
    moves = [] # (x, y), piece, mask

    pieces = getPieces(board, filter)

    for position, piece in pieces:
        moves.append((position, piece, PieceDefs.VALUE_MAP[Piece.extractPiece(piece)].getValidMoves(board, board.getI(position[0], position[1]), onlyCapturing)))

    return moves
def getChecks(board, kingPosition):
    isWhite = Piece.isWhite(board.getAt2D(kingPosition[0], kingPosition[1]))

    moves = getMoves(board, True, Piece.BLACK_VALUE if isWhite else Piece.WHITE_VALUE)

    checkSources = []

    attackedMask = ChessBoardMask()
    for position, piece, moves in moves:
        if(moves.getAt2D(kingPosition[0], kingPosition[1])):
            checkSources.append((position, piece, moves))

        attackedMask = attackedMask.getOr(moves)

    return (checkSources, attackedMask)

def getKingState(board, kingPosition):
    isWhite = Piece.isWhite(board.getAt2D(kingPosition[0], kingPosition[1]))

    checkSources, attackedMask = getChecks(board, kingPosition)
    
    kingMoves = PieceDefs.KING.getValidMoves(board, board.getI(kingPosition[0], kingPosition[1]))

    if(len(checkSources) > 0 and attackedMask.getIncludes(kingMoves)):
        for position, piece, moves in getMoves(board, False, Piece.WHITE_VALUE if isWhite else Piece.BLACK_VALUE):
            if(position == kingPosition):
                continue

            for movePosition in moves.validPositions():
                checkSources, _ = getChecks(board.copyWithMove(position[0], position[1], movePosition[0], movePosition[1]), kingPosition)

                if(len(checkSources) <= 0): # found a line without check
                    return (True, False)
        return (True, True)
    elif(len(checkSources) > 0):
        return (True, False) # Checked, Not Checkmated
    else:
        return (False, False) # Not Checked, Not Checkmated

In [ ]:
board = ChessBoardState()

kingPos = (0,0)

board.setAt2D(kingPos[0], kingPos[1], PieceDefs.KING.getValue(True))
board.setAt2D(0, 1, PieceDefs.KNIGHT.getValue(True))

board.setAt2D(7, 0, PieceDefs.ROOK.getValue(False))
board.setAt2D(7, 1, PieceDefs.ROOK.getValue(False))
board.setAt2D(0, 6, PieceDefs.ROOK.getValue(False))

print(board)

print("Status:", getKingState(board, kingPos))

     ┌───┬───┬───┬───┬───┬───┬───┬───┐
  8  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  7  │ r │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  6  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  5  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  4  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  3  │   │   │   │   │   │   │   │   │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  2  │ N │   │   │   │   │   │   │ r │
     ├───┼───┼───┼───┼───┼───┼───┼───┤
  1  │ K │   │   │   │   │   │   │ r │
     └───┴───┴───┴───┴───┴───┴───┴───┘
       a   b   c   d   e   f   g   h  
Status: (True, True)
